# INV-M365-DK - Standalone Graph Runtime Integration Pack - P0A Current Artifact Autopsy

**Plan reference:** `plan:m365-standalone-graph-runtime-integration-pack:R1`  
**Phase:** `P0A` - Current artifact autopsy.  
**Purpose:** Prove with deterministic file inspection that the current marketplace artifact is a UCP-facing client/contract pack and does not contain a Microsoft Graph runtime service, secure auth, token storage, or self-contained launch behavior. This notebook backs the governance autopsy doc and machine-readable verification under Phase `P0A`.

**Inputs:**

- `dist/m365_pack/manifest.json`
- `dist/m365_pack/payload.tar.gz`
- `dist/m365_pack/com.smarthaus.m365-1.0.0.ucp.tar.gz`
- `dist/m365_pack/evidence/conformance.json`
- `IntegrationPacks/M365/1.0.0/manifest.json`
- `IntegrationPacks/M365/1.0.0/SHA256SUMS`
- `IntegrationPacks/M365/1.0.0/conformance.json`
- `IntegrationPacks/M365/1.0.0/provenance.json`
- `src/ucp_m365_pack/setup_schema.json`
- `src/ucp_m365_pack/client.py`
- `src/m365_server/__main__.py`

**Output:**

- `configs/generated/standalone_graph_runtime_integration_pack_p0_current_artifact_autopsy_v1_verification.json`

**Determinism:** Reads-only over the committed bundle, the installed Integration Pack, and the source files. No network. No secrets. Reproducible across runs of the same commit.

**Invariants asserted:**

- I1 - The packaged payload contains the UCP client + contracts + setup schema + an `agents.yaml` registry file, and does not contain a Microsoft Graph runtime service, auth flow, token store, launcher, or Graph client implementation.
- I2 - The setup schema requires `M365_OPS_ADAPTER_URL` and a JWT minting secret, which means readiness depends on an external service not packaged inside the artifact.
- I3 - The UCP-facing `client.py` resolves the M365 source repo via `M365_REPO_ROOT` env and parent-directory walks for `registry/agents.yaml` and `src/ops_adapter/`, which violates the no-source-repo rule when invoked from an installed artifact.
- I4 - The current `m365_server` launcher imports `from ops_adapter.main import app` and walks `UCP_ROOT` / `REPOS_ROOT` for tenants, which means the runtime cannot launch without the M365 source tree on the host.
- I5 - There is no Microsoft OAuth (Authorization Code + PKCE or device-code) implementation, no app-only client-secret/certificate flow, no Keychain-backed token store, and no Graph client with retry/throttle/normalized error classes inside the packaged payload.

In [ ]:
import hashlib
import json
import os
import re
import tarfile
from pathlib import Path

REPO = Path("/Users/smarthaus/Projects/GitHub/M365").resolve()
PACKS = Path("/Users/smarthaus/Projects/GitHub/IntegrationPacks/M365/1.0.0").resolve()

manifest = json.loads((REPO / "dist/m365_pack/manifest.json").read_text())
setup_schema = json.loads((REPO / "src/ucp_m365_pack/setup_schema.json").read_text())
client_src = (REPO / "src/ucp_m365_pack/client.py").read_text()
server_src = (REPO / "src/m365_server/__main__.py").read_text()
conformance = json.loads((REPO / "dist/m365_pack/evidence/conformance.json").read_text())
provenance = json.loads((PACKS / "provenance.json").read_text())

with tarfile.open(REPO / "dist/m365_pack/payload.tar.gz") as tf:
    payload_members = sorted(m.name for m in tf.getmembers() if m.isfile())
with tarfile.open(REPO / "dist/m365_pack/com.smarthaus.m365-1.0.0.ucp.tar.gz") as tf:
    bundle_members = sorted(m.name for m in tf.getmembers() if m.isfile())

(payload_members, bundle_members[:6])

In [ ]:
# Test I1: payload contains client+contracts+setup+registry, but no real graph runtime service.
expected_payload = {
    "registry/agents.yaml",
    "setup_schema.json",
    "ucp_m365_pack/__init__.py",
    "ucp_m365_pack/client.py",
    "ucp_m365_pack/contracts.py",
}
missing_required = expected_payload - set(payload_members)
extra_payload = set(payload_members) - expected_payload
graph_runtime_indicators = [
    p for p in payload_members
    if any(token in p for token in (
        "m365_server/", "smarthaus_graph/", "ops_adapter/", "oauth", "pkce",
        "keychain", "auth/", "graph_client",
    ))
]
assert not missing_required, missing_required
assert not extra_payload, extra_payload
assert not graph_runtime_indicators, graph_runtime_indicators
i1_ok = True
i1_ok

In [ ]:
# Test I2: setup schema requires external ops adapter URL + JWT secret, no Graph auth fields packaged.
required = set(setup_schema.get("required", []))
props = setup_schema.get("properties", {})
graph_auth_fields = {
    "GRAPH_TENANT_ID", "GRAPH_CLIENT_ID", "GRAPH_CLIENT_SECRET",
    "M365_TENANT_ID", "M365_CLIENT_ID", "M365_AUTH_MODE",
    "M365_REDIRECT_URI", "M365_DEVICE_CODE_FALLBACK",
    "M365_APP_ONLY_CERTIFICATE_REF", "M365_APP_ONLY_CLIENT_SECRET_REF",
    "M365_TOKEN_STORE", "M365_KEYCHAIN_SERVICE",
}
i2_ok = (
    "M365_OPS_ADAPTER_URL" in required
    and "M365_SERVICE_JWT_HS256_SECRET" in required
    and "M365_SERVICE_ACTOR_UPN" in required
    and not (graph_auth_fields & set(props.keys()))
)
i2_ok

In [ ]:
# Test I3: client.py walks the source tree for registry + ops_adapter and reads M365_REPO_ROOT.
i3_indicators = {
    "M365_REPO_ROOT_env": "M365_REPO_ROOT" in client_src,
    "resolve_m365_repo_root_fn": "_resolve_m365_repo_root" in client_src,
    "registry_agents_yaml_lookup": "registry\" / \"agents.yaml" in client_src or "registry / agents.yaml" in client_src,
    "src_ops_adapter_lookup": "src\" / \"ops_adapter" in client_src or "src / ops_adapter" in client_src,
    "http_service_required": "M365_OPS_ADAPTER_URL" in client_src,
    "direct_import_unavailable": '"direct_import_available": False' in client_src or "'direct_import_available': False" in client_src,
}
i3_ok = all(i3_indicators.values())
i3_ok, i3_indicators

In [ ]:
# Test I4: m365_server launcher relies on local source tree and UCP/REPOS roots, not the installed pack.
i4_indicators = {
    "imports_ops_adapter_main": "from ops_adapter.main import app" in server_src,
    "reads_M365_APP_ROOT": "M365_APP_ROOT" in server_src,
    "requires_registry_agents_yaml": "registry\" / \"agents.yaml" in server_src or "registry / agents.yaml" in server_src,
    "reads_UCP_ROOT_or_REPOS_ROOT": ("UCP_ROOT" in server_src) or ("REPOS_ROOT" in server_src) or ("UCP_REPOS_ROOT" in server_src),
    "sibling_repo_walk": "app_root.parent / \"UCP\" / \"tenants\"" in server_src or "app_root.parent / 'UCP' / 'tenants'" in server_src,
}
i4_ok = all(i4_indicators.values())
i4_ok, i4_indicators

In [ ]:
# Test I5: no Graph OAuth, app-only, Keychain, retry/throttle, or Graph error normalization in payload.
search_tokens = [
    "authorization_code", "pkce", "device_code", "client_credentials",
    "keychain", "keyring", "msal", "adal",
    "throttle", "retry-after", "x-ms-throttle",
    "GraphClient", "graph.microsoft.com",
]
missing_in_payload = []
with tarfile.open(REPO / "dist/m365_pack/payload.tar.gz") as tf:
    text_blob = b""
    for member in tf.getmembers():
        if member.isfile():
            f = tf.extractfile(member)
            if f is not None:
                text_blob += f.read()
    decoded = text_blob.decode("utf-8", errors="ignore").lower()
    for token in search_tokens:
        if token.lower() not in decoded:
            missing_in_payload.append(token)
i5_ok = len(missing_in_payload) == len(search_tokens)
i5_ok, missing_in_payload

In [ ]:
# Emit verification artifact.
verification = {
    "plan_ref": "plan:m365-standalone-graph-runtime-integration-pack:R1",
    "phase": "P0A",
    "purpose": "Current artifact autopsy: prove the marketplace artifact is a UCP client/contract pack, not a real Microsoft Graph integration runtime.",
    "manifest": {
        "pack_id": manifest["pack_id"],
        "version": manifest["version"],
        "distribution_mode": manifest["distribution_mode"],
        "capabilities_exposed": manifest["capabilities_exposed"],
        "setup_schema_ref": manifest["setup_schema_ref"],
        "entrypoint": manifest["entrypoint"],
    },
    "payload_inventory": payload_members,
    "bundle_inventory": bundle_members,
    "setup_required_fields": sorted(setup_schema.get("required", [])),
    "setup_property_fields": sorted(setup_schema.get("properties", {}).keys()),
    "client_repo_root_env": ["M365_REPO_ROOT", "SMARTHAUS_M365_REPO_ROOT"],
    "client_repo_root_walk": ["registry/agents.yaml", "src/ops_adapter"],
    "server_sibling_lookup_env": ["M365_APP_ROOT", "UCP_ROOT", "REPOS_ROOT", "UCP_REPOS_ROOT"],
    "server_imports_ops_adapter": "from ops_adapter.main import app",
    "missing_runtime_indicators": [
        "m365_server packaged inside payload",
        "smarthaus_graph packaged inside payload",
        "ops_adapter packaged inside payload",
        "OAuth Authorization Code + PKCE flow",
        "Device-code fallback",
        "App-only client secret or certificate flow",
        "macOS Keychain or encrypted token store",
        "Graph client with retry / throttle / normalized errors",
    ],
    "invariants": {
        "I1_payload_only_contracts": i1_ok,
        "I2_setup_external_service_only": i2_ok,
        "I3_client_walks_source_tree": i3_ok,
        "I4_server_requires_source_tree": i4_ok,
        "I5_no_graph_runtime_in_payload": i5_ok,
    },
    "conformance_local": conformance["status"],
    "sha256sums_match_provenance": provenance["artifact"]["bundle_sha256"] == hashlib.sha256(
        (PACKS / "com.smarthaus.m365-1.0.0.ucp.tar.gz").read_bytes()
    ).hexdigest(),
    "verdict": "current_artifact_is_ucp_client_contract_pack_only",
    "truthful": all([i1_ok, i2_ok, i3_ok, i4_ok, i5_ok]),
}
out = REPO / "configs/generated/standalone_graph_runtime_integration_pack_p0_current_artifact_autopsy_v1_verification.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(verification, indent=2, sort_keys=True))
verification["verdict"], verification["truthful"]